# Data Cleaning Step

The purpose of this notebook is to document the steps to clean up the dataset to prepare it for future analysis. The notebook will follow the following cleaning steps:

1. coerce passenger count and ratecodeID into integer data types
2. remove those rows with questionable pick up and drop off times
3. remove the rows where the drop off time is earlier than the pick up time
4. remove the rows where the pick up time is outside of Janurary
5. mark the rows where there are negative totals
6. mark the rows where there are negative totals and there are reversal positive matching rows

In [1]:
# Imports

import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Read in the files

df = pd.read_parquet("../data/raw/yellow_tripdata_2026-01.parquet")
taxi_zone = pd.read_csv("../data/raw/taxi_zone_lookup.csv")

In [4]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [5]:
taxi_zone.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### Step 1: Coerce Passenger Count into Integer Data Types

In [7]:
df.dtypes.reset_index()

,index,0
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[us]
2,tpep_dropoff_datetime,datetime64[us]
3,passenger_count,float64
4,trip_distance,float64
5,RatecodeID,float64
6,store_and_fwd_flag,str
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


In [15]:
df['RatecodeID'] = df['RatecodeID'].astype('Int32')
df[['RatecodeID']].sample(10)

,RatecodeID
177583,1
263016,1
3100225,<NA>
2221973,1
3451379,<NA>
2009546,1
402353,1
3494043,<NA>
1415769,1
1994265,1


In [16]:
df['passenger_count'] = df['passenger_count'].astype('Int32')
df[['passenger_count']].sample(10)

,passenger_count
3370924,<NA>
2330333,1
606883,1
3706167,<NA>
2199000,1
973907,1
1120261,1
2337498,1
482839,1
2666199,<NA>


### Step 2: Remove columns with questionable pick up times (dropff time that is earlier thatn pick up time)

In [17]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1465434,1,2026-01-17 19:30:00,2026-01-17 19:18:18,1,16.5,99,N,46,86,1,66.5,0.0,0.5,0.0,0.0,0.0,67.0,0.0,0.0,0.0


In [20]:
df.drop(index=1465434, inplace=True)

In [21]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


this row has been removed

## Step 3: Remove the rows where the pick up time is earlier than Jaurary

In [28]:
start_date = pd.Timestamp("2026-01-01")
outside_bound_list = df[df['tpep_pickup_datetime'] < start_date].index.tolist()
outside_bound_list

[980, 981, 982, 1404, 2289, 5181]

In [29]:
df.drop(index=outside_bound_list, inplace=True)

In [30]:
df[df['tpep_dropoff_datetime'] < start_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


In [32]:
end_date = pd.Timestamp('2026-02-01')
outside_bound_list = df[df['tpep_pickup_datetime'] >= end_date].index.tolist()
outside_bound_list

[2635466]

In [34]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2635466,2,2026-02-01 00:45:01,2026-02-01 01:03:08,1,1.78,1,N,79,249,1,15.6,1.0,0.5,5.34,0.0,1.0,26.69,2.5,0.0,0.75


In [35]:
df.drop(index=outside_bound_list, inplace=True)

In [36]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


pick up times that are later than end date are also dropped

### Step 4: Mark the rows where the totals are negative and also if there are reversal transactions

In [38]:
df['is_negative_total'] = df['total_amount'] < 0
df[['is_negative_total']].sample(10)

,is_negative_total
3468760,False
2072626,False
2009471,False
2818668,False
163302,False
2758201,False
853036,False
1712700,False
996966,False
1318378,False


In [39]:
df['is_matched_reversal'] = False

In [41]:
match_cols = [
    "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
]

negative_trips = df[df['is_negative_total'] == True]
positive_trips = df[df['is_negative_total'] == False]

matched_pairs = negative_trips.merge(positive_trips, on=match_cols, suffixes=("_neg", "_pos"))
matched_pairs.shape

(35712, 39)

In [42]:
matched_pairs.sample(10)

,VendorID_neg,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count_neg,trip_distance,RatecodeID_neg,store_and_fwd_flag_neg,PULocationID,DOLocationID,payment_type_neg,...,mta_tax_pos,tip_amount_pos,tolls_amount_pos,improvement_surcharge_pos,total_amount_pos,congestion_surcharge_pos,Airport_fee_pos,cbd_congestion_fee_pos,is_negative_total_pos,is_matched_reversal_pos
15691,2,2026-01-14 11:19:33,2026-01-14 11:51:30,1,8.45,1,N,132,197,4,...,0.5,0.0,0.00,1.0,41.95,0.0,1.75,0.00,False,False
25693,2,2026-01-22 17:44:21,2026-01-22 18:06:34,2,4.17,1,N,186,261,3,...,0.5,0.0,0.00,1.0,30.55,2.5,0.00,0.75,False,False
34769,2,2026-01-31 09:17:35,2026-01-31 09:42:06,2,4.84,1,N,48,45,4,...,0.5,0.0,0.00,1.0,31.55,2.5,0.00,0.75,False,False
4085,2,2026-01-04 07:16:17,2026-01-04 07:23:42,1,5.18,1,N,132,130,4,...,0.5,0.0,0.00,1.0,24.45,0.0,1.75,0.00,False,False
34720,2,2026-01-31 07:59:27,2026-01-31 08:14:55,1,3.17,1,N,90,231,4,...,0.5,0.0,0.00,1.0,22.45,2.5,0.00,0.75,False,False
7107,2,2026-01-06 21:39:42,2026-01-06 21:51:58,1,1.69,1,N,100,233,4,...,0.5,0.0,0.00,1.0,18.55,2.5,0.00,0.75,False,False
33939,2,2026-01-30 17:42:22,2026-01-30 17:42:34,1,0.03,2,N,142,142,4,...,0.5,0.0,0.00,1.0,79.00,2.5,0.00,0.00,False,False
20604,2,2026-01-18 06:47:12,2026-01-18 07:04:56,1,6.79,1,N,186,255,3,...,0.5,0.0,7.46,1.0,41.11,2.5,0.00,0.75,False,False
32382,2,2026-01-29 13:52:03,2026-01-29 14:01:55,1,1.10,1,N,237,236,2,...,0.5,0.0,0.00,1.0,14.00,2.5,0.00,0.00,False,False
33708,2,2026-01-30 14:27:06,2026-01-30 14:29:00,1,0.12,1,N,238,238,2,...,0.0,0.0,0.00,0.0,2.50,2.5,0.00,0.00,False,False


In [44]:
df["reversal_key"] = list(
    zip(
        df["tpep_pickup_datetime"],
        df["tpep_dropoff_datetime"],
        df["PULocationID"],
        df["DOLocationID"],
        df["trip_distance"],
        df["total_amount"]
    )
)

matched_negative_keys = set(
    zip(
        matched_pairs["tpep_pickup_datetime"],
        matched_pairs["tpep_dropoff_datetime"],
        matched_pairs["PULocationID"],
        matched_pairs["DOLocationID"],
        matched_pairs["trip_distance"],
        matched_pairs["total_amount_neg"]
    )
)

df['is_matched_reversal'] = df['reversal_key'].isin(matched_negative_keys)

In [45]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal,reversal_key
2894970,2,2026-01-10 08:15:11,2026-01-10 08:20:02,<NA>,1.11,<NA>,NaN,71,71,0,...,0.00,0.0,1.0,11.84,NaN,NaN,0.00,False,False,"(2026-01-10 08:15:11, 2026-01-10 08:20:02, 71,..."
364684,1,2026-01-05 22:32:59,2026-01-05 22:43:28,4,1.50,1,N,148,249,1,...,2.00,0.0,1.0,19.85,2.5,0.0,0.75,False,False,"(2026-01-05 22:32:59, 2026-01-05 22:43:28, 148..."
3604915,2,2026-01-29 21:21:30,2026-01-29 21:29:45,<NA>,0.54,<NA>,NaN,234,234,0,...,0.00,0.0,1.0,18.94,NaN,NaN,0.75,False,False,"(2026-01-29 21:21:30, 2026-01-29 21:29:45, 234..."
3459474,2,2026-01-26 09:19:45,2026-01-26 09:28:10,<NA>,1.36,<NA>,NaN,249,144,0,...,0.00,0.0,1.0,22.58,NaN,NaN,0.75,False,False,"(2026-01-26 09:19:45, 2026-01-26 09:28:10, 249..."
2629118,2,2026-01-31 22:04:25,2026-01-31 22:19:20,1,0.93,1,N,48,100,1,...,3.99,0.0,1.0,23.94,2.5,0.0,0.75,False,False,"(2026-01-31 22:04:25, 2026-01-31 22:19:20, 48,..."
52401,1,2026-01-01 18:27:03,2026-01-01 18:34:22,1,1.10,1,N,262,75,2,...,0.00,0.0,1.0,12.60,2.5,0.0,0.00,False,False,"(2026-01-01 18:27:03, 2026-01-01 18:34:22, 262..."
3722956,2,2026-01-31 23:16:13,2026-01-31 23:19:48,<NA>,0.54,<NA>,NaN,236,237,0,...,0.00,0.0,1.0,11.90,NaN,NaN,0.00,False,False,"(2026-01-31 23:16:13, 2026-01-31 23:19:48, 236..."
3312588,2,2026-01-22 12:15:00,2026-01-22 12:31:00,<NA>,6.62,<NA>,NaN,244,142,0,...,0.00,0.0,1.0,28.52,NaN,NaN,0.75,False,False,"(2026-01-22 12:15:00, 2026-01-22 12:31:00, 244..."
250245,2,2026-01-04 13:17:49,2026-01-04 13:41:27,1,7.02,1,N,237,231,1,...,5.50,0.0,1.0,43.35,2.5,0.0,0.75,False,False,"(2026-01-04 13:17:49, 2026-01-04 13:41:27, 237..."
3474326,2,2026-01-26 19:08:36,2026-01-26 19:17:51,<NA>,0.00,<NA>,NaN,236,142,0,...,0.00,0.0,1.0,18.93,NaN,NaN,0.00,False,False,"(2026-01-26 19:08:36, 2026-01-26 19:17:51, 236..."


In [46]:
df.drop(columns=['reversal_key'], inplace=True)

In [47]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal
3647587,2,2026-01-30 19:00:18,2026-01-30 19:44:26,<NA>,14.70,<NA>,NaN,14,129,0,...,0.5,0.00,0.0,1.0,65.96,NaN,NaN,0.00,False,False
3037217,1,2026-01-14 18:14:17,2026-01-14 18:20:55,<NA>,0.60,<NA>,NaN,140,237,0,...,0.5,1.00,0.0,1.0,14.95,NaN,NaN,0.00,False,False
1127226,1,2026-01-14 13:35:14,2026-01-14 13:57:38,1,2.60,1,N,238,141,1,...,0.5,3.00,0.0,1.0,26.85,2.5,0.0,0.75,False,False
2027932,1,2026-01-23 23:20:15,2026-01-23 23:26:03,1,0.70,1,N,141,236,1,...,0.5,2.90,0.0,1.0,14.40,2.5,0.0,0.00,False,False
111998,2,2026-01-02 16:28:51,2026-01-02 16:39:02,3,1.27,1,N,68,114,2,...,0.5,0.00,0.0,1.0,17.95,2.5,0.0,0.75,False,False
2756901,2,2026-01-04 18:09:49,2026-01-04 18:29:58,<NA>,5.72,<NA>,NaN,148,226,0,...,0.5,0.00,0.0,1.0,33.51,NaN,NaN,0.75,False,False
370349,1,2026-01-06 06:42:54,2026-01-06 07:11:27,1,6.10,99,N,49,246,1,...,0.5,0.00,0.0,0.0,29.00,0.0,0.0,0.00,False,False
3559517,2,2026-01-28 21:08:15,2026-01-28 21:23:30,<NA>,1.76,<NA>,NaN,162,143,0,...,0.5,0.00,0.0,1.0,93.96,NaN,NaN,0.75,False,False
46305,2,2026-01-01 16:11:26,2026-01-01 16:41:47,1,1.53,1,N,162,230,2,...,0.5,0.00,0.0,1.0,31.95,2.5,0.0,0.75,False,False
1595218,2,2026-01-19 15:39:55,2026-01-19 15:50:19,1,1.60,1,N,48,90,1,...,0.5,3.23,0.0,1.0,19.38,2.5,0.0,0.75,False,False


In [49]:
df.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'is_negative_total',
 'is_matched_reversal']